### Application 2: Aspect-based summariser
Generates an aspect-based summary of the student feedback comments in the feedback dataset. 
Based on the identified aspects within the given feedback, the user is prompted to choose one aspect within this list of aspects. Qwen 3.5 Plus generates a summary about this aspect of the instructors teaching.

To run, activate the `openai_environment` conda environment specified in the environements directory. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [ ]:
# sample data

feedback_dataset = [
    "The assignments were well-structured and really helped reinforce the concepts taught in lectures.",
    "I felt that the assignments were too time-consuming compared to their contribution to the final grade.",
    "The weekly assignments kept me consistent with the module content and prevented last-minute cramming.",
    "Some assignments felt repetitive and did not add much new learning beyond the tutorials.",
    "The assignments were challenging but fair, and I learned a lot by working through them.",
    "I appreciated how the assignments gradually increased in difficulty throughout the semester.",
    "The assignment instructions were often unclear, which made it stressful to know what was expected.",
    "The assignments aligned closely with the exam format, which made revision much easier.",
    "While the assignments were interesting, the workload was overwhelming when combined with other modules.",
    "The open-ended nature of the assignments encouraged deeper thinking and creativity.",
    "I struggled with the assignments at the start because there were not enough worked examples provided.",
    "The assignments helped me identify my weak areas early in the semester.",
    "Feedback for the assignments was detailed and helped me improve in later submissions.",
    "Some assignments felt disconnected from the lecture content and required a lot of self-learning.",
    "The assignments were practical and gave me hands-on experience applying theoretical concepts.",
    "I found the assignment deadlines too closely spaced, especially during midterm period.",
    "The assignments were engaging and made the module more enjoyable overall.",
    "Grading for the assignments felt inconsistent across different markers.",
    "The assignments pushed me to think critically rather than just apply formulas.",
    "I appreciated that the assignments encouraged collaboration while still being individually assessed.",
    "The difficulty level of the assignments was higher than expected for this module.",
    "The assignments were manageable and well-paced across the semester.",
    "Some assignments felt more like busywork than meaningful learning activities.",
    "The assignments helped bridge the gap between theory and real-world applications.",
    "I often spent more time interpreting the assignment requirements than solving the actual problems.",
    "The assignments were rewarding, especially when I could see my improvement over time.",
    "The scope of the assignments was too broad for the given time frame.",
    "I liked that the assignments allowed multiple approaches rather than a single correct answer.",
    "The assignments were stressful but ultimately prepared me well for the final assessment.",
    "Overall, the classes were a valuable component of the module and enhanced my understanding.",
    "The instructor maintained a good pace throughout the lectures, making it easy to follow along.",
    "I felt that the lectures were too fast, especially when new concepts were introduced.",
    "The pacing of the class was well-balanced and allowed enough time to understand each topic.",
    "Sometimes the instructor moved too quickly through important concepts without sufficient explanation.",
    "The pace was appropriate and kept the lectures engaging without feeling rushed.",
    "I struggled to keep up because the instructor covered too much content in a short time.",
    "The instructor adjusted the pace based on student feedback, which was very helpful.",
    "The lectures occasionally felt slow, particularly during revision of basic concepts.",
    "I appreciated that the instructor paused regularly to check for understanding.",
    "The pace of the lectures was inconsistent, with some sessions feeling rushed and others too slow.",
    "The instructor went through examples at a manageable speed, which helped reinforce learning.",
    "There was not enough time given to absorb complex topics before moving on.",
    "The pacing was ideal for someone with prior knowledge, but difficult for beginners.",
    "I found the lectures too slow at times, which made it hard to stay focused.",
    "The instructor maintained a steady pace that made the content easy to digest.",
    "Key concepts were sometimes rushed, making it difficult to fully understand them.",
    "The instructor took time to explain difficult ideas, which balanced out the overall pace.",
    "The lectures felt too fast, especially when covering technical material.",
    "I liked that the instructor revisited previous topics, which helped manage the pace.",
    "The pacing was too slow for me, as too much time was spent on simple examples.",
    "The instructor managed the pace well, ensuring most students could follow along.",
    "There were moments where the lecture speed increased significantly, making it hard to keep up.",
    "The pace allowed enough time for note-taking and asking questions.",
    "I often felt rushed during lectures and could not fully process the material.",
    "The instructor paced the lessons effectively, balancing theory and examples.",
    "The lectures were sometimes too slow, especially when repeating previously covered material.",
    "The instructor spoke and presented at a pace that was easy to follow.",
    "The pace was too fast when introducing new topics, but slower during revisions.",
    "I appreciated the consistent pacing across all lectures.",
    "Overall, the instructor’s pacing supported my understanding of the module."
]

In [ ]:
from openai import OpenAI
import ast

# maps an aspect to a list of indices of comments
# that contain the aspect
aspect_map = {}

# instantiate api connection for gpt-5.2
client = OpenAI(
  api_key="API-KEY"
)

# extract aspects from each feedback comment 
# and populate aspect_map  
for i in range(len(feedback_dataset)):
    comment = feedback_dataset[i]
    extract_aspects_prompt = f"""
    Extract the key aspects from the given student feedback comment.
    Rules:
    - Return ONLY a valid Python list of strings.
    - Each aspect MUST be a single world lemma.
    - ALL elements in the list must be quoted.
    Example output:
    ["pace", "clarity"]
    Comment: {comment}
    """

    raw_sentiment_output = client.responses.create(
        model="gpt-5.2",
        input = extract_aspects_prompt,
        store=True,
    ).output_text

    # extracted list of aspect for comment
    aspect_list = ast.literal_eval(raw_sentiment_output)

    for aspect in aspect_list:
        if aspect not in aspect_map:
            aspect_map[aspect] = []
        if i not in aspect_map[aspect]:
            aspect_map[aspect].append(i)	

In [8]:
# prompting instructors to choose from list of covered aspects
formatted_aspect_list = "\n".join(aspect_map.keys())
aspect = input(f"""The following aspects of your course have been addressed in your students' feedback comments. 
               Out of the listed aspects, please enter the aspect you would like to analyse.\n{formatted_aspect_list}""")

In [9]:
# retrieving and formatting relevant comments
relevant_comment_indices = aspect_map[aspect]
formatted_comment_list = ""
index = 1

for comment_index in relevant_comment_indices:
    retrieved_comment = feedback_dataset[comment_index]
    formatted_comment_list += f"{index}." + retrieved_comment + " "
    index += 1


In [ ]:
# instantiate api connection for qwen 3.5 plus
client = OpenAI(
    api_key="API-KEY",
    base_url="https://dashscope-intl.aliyuncs.com/api/v2/apps/protocols/compatible-mode/v1",
)

summary_prompt = prompt = f"""You are generating a high-quality summary of student feedback.
Below is an example of how 10 student feedback comments should be aggregated into one summary.

Example:
Comments:
1. Course difficulty will vary among professors. I took it with Sara Humphreys. I did not have much to take home from the lectures, and the assigned work felt made the course feel like a very upper year / graduate studies course. Dropped and retook it online with Randy Harris and not only was the course assignment less painful, I also learned more and had a very enjoyable time.
2. I loved Sara Humphreys! she was amazing and she's very smart. knows how to teach and explains the material veryyy well! i took her class during the summer semester and got a 97%. Deff recommend her to anyone going into the medical field:)
3. Not a hard course if you go to class and listen to the prof( Bill Bishop). Too many slides to cram, but they are a strong resource, especially if you annotate them during lectures. Plenty of past midterms and exams but don't expect a really high mark just from those. A lot of content that can be dry at times, but overall an enjoyable course. Labs are very long and can go wrong just about anywhere if not set up properly so get help from the TA's while you can.
4. Like state previously, Bill Bishop is a bad teacher but a nice person. He means well, but his lectures make no sense at all. I took him for both acct classes and did not learn a single thing. His lectures relate to nothing to accounting and they are a waste of time to go to. Everything is online and open book, open notes and easy grading.
5. Great place, great university.
6. ULTRA BORING! dont even register unless you know you want to be bored out of your mind. The class does not seem that hard, but the readings are dumb hard to understand and he sounds so uninterested in the class. oh and he says " right " at the end of each sentence. so annoying
7. Professor Lawton explains things well and shows why they matter in real life. He did a great job making complicated concepts seem simple. He assigns a lot of homework, but not more than any other college-level calculus class.
8. Professor Marney is great! She gave me a lot of help on my papers and was willing to work with me when I had to miss class because of a long illness. She is nice and her class is really interesting. If you can take her class, you should.
9. The most useless course I've ever taken. The standards for the assignments are ridiculously low. You'd think in a design class, you'd learn how to design / plan / implement projects were economically or, at very least, physically feasible, but that really wasn't the case. Many of the groups that retrieved high marks had issues with hindering their entire designs on technologies that did not yet exist, were clearly economically unfeasible, or didn't really solve the problem they were tasked to solve. The first two out of three major projects were designing a poster and re-design of a room/ space. The course really wasn't challenging, didn't introduce any novel material about design or design strategies, and no real concern seemed to go towards the viability of designs / solutions.
10. If you've done AP or IB physics in high school, this shouldn't be too bad. Understanding the concepts is absolutely essential: if you just try to memorize problems or equations, you will fail.

Summary:
Student feedback shows that course experiences vary significantly by instructor. With Sara Humphreys, one student found the lectures unhelpful and assignments graduate-level, while another praised her clear explanations and strong teaching. A student who retook the course with Randy Harris described the assignments as less painful and the learning experience as more enjoyable. Views on Bill Bishop also conflicted: some found the course manageable if lectures were attended, noting extensive slides and past exams as useful resources, while others felt his lectures made little sense despite easy open-book grading. Engagement differed across classes, with some describing material as dry or boring and readings difficult, while Professor Lawton was praised for making complex calculus concepts clear and relevant. Professor Marney was commended for being supportive, especially during illness. Workload ranged from long, error-prone labs to manageable homework. A design course was criticised for low standards and unrealistic project expectations. One reviewer simply praised the university overall. Furthermoer, one comment emphasised the importance of understanding concepts rather than memorising formulas in their course.

Instructions:
Now summarise the following sentences into 1 summary with a similar approach based on the aspect {aspect}. 

Requirements: 
Accuracy:
- Use only information that is explicitly mentioned in the comments.
- Do not invent trends and hallucinate. The summary must be supported by evidence in the comments.
- If there are conflicting comments, display both sides instead of forcing agreement.

Coherence:
- Summary is well-structured and easy to follow.
- There is smooth transition between ideas. 

Coverage:
- All key ideas and major themes are mentioned in the summary.
- Combine similar points into unified themes instead of listing them individually.

Output format:
1) Executive summary (140-180 words)

Sentences:
{formatted_comment_list}"""

response = client.responses.create(
    model="qwen3.5-plus",
    input=summary_prompt,
    extra_body={
        "enable_thinking": True
    }
)

print(f"Below is a summary of student opinons towards the aspect {aspect}")
print(response.output[1].content[0].text)

Below is a summary of student opinons towards the aspect pace
Executive Summary:
Student feedback regarding course pace reveals mixed experiences. Many students praised the instructor for maintaining a steady, well-balanced pace that facilitated understanding and allowed sufficient time for note-taking and questions. Several comments highlighted that assignments were manageable and well-paced throughout the semester. The instructor was commended for adjusting speed based on feedback and revisiting previous topics to aid comprehension. However, conflicting views emerged regarding lecture speed. A significant number of students felt the pace was too fast, particularly when introducing new or technical concepts, causing them to struggle keeping up or processing material. Conversely, others found the lectures too slow, especially during revisions of basic concepts or simple examples, which made staying focused difficult. Some described the pacing as inconsistent, varying between rushed and